# Evaluating self-supervised representations

_Semi & Self-Supervised Learning_

**You trained an encoder on unlabeled data. How do you actually tell if its representation is any good?**

Self-supervised training gives you an encoder $f$: it turns a raw input $x$ into a feature vector $z=f(x)$. There were no labels in training, so accuracy never appeared. The question remains: is $z$ a good representation?

       "Good" means the features make the real downstream task easy — same-class inputs land near each other, classes are separable, and few labels are needed to read the answer off the features.

---

This notebook is a practice scaffold for the **Evaluating self-supervised representations** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----- a pretrained, FROZEN encoder (from SimCLR / MoCo / etc.) -----
encoder = build_pretrained_encoder()      # e.g. a ResNet trunk, returns (B, d) features
encoder.eval()
for p in encoder.parameters():            # FREEZE: no gradients flow into the encoder
    p.requires_grad_(False)

device = "cuda"
encoder.to(device)

@torch.no_grad()
def extract_features(loader):
    """Run the frozen encoder once; cache features + labels."""
    feats, labels = [], []
    for x, y in loader:
        z = encoder(x.to(device))         # (B, d) representation
        feats.append(z.cpu()); labels.append(y)
    return torch.cat(feats), torch.cat(labels)

Ztr, ytr = extract_features(train_loader)  # frozen features for the labeled set
Zte, yte = extract_features(test_loader)

# =================== 1) LINEAR PROBE ===================
num_classes = int(ytr.max()) + 1
probe = nn.Linear(Ztr.size(1), num_classes).to(device)   # the ONLY trainable weights
opt = torch.optim.SGD(probe.parameters(), lr=1e-2, momentum=0.9, weight_decay=0.0)

Ztr_d, ytr_d = Ztr.to(device), ytr.to(device)
for epoch in range(100):                  # train only the linear head on frozen features
    opt.zero_grad()
    logits = probe(Ztr_d)                  # W . z + b
    loss = F.cross_entropy(logits, ytr_d)
    loss.backward()
    opt.step()

with torch.no_grad():
    pred = probe(Zte.to(device)).argmax(1).cpu()
    linear_probe_acc = (pred == yte).float().mean().item()
print(f"linear-probe accuracy: {linear_probe_acc:.3f}")

# =================== 2) kNN EVALUATION ===================
def knn_eval(Ztr, ytr, Zte, yte, k=20, temperature=0.07):
    """Cosine-similarity weighted k-NN on FROZEN features (DINO-style)."""
    Ztr = F.normalize(Ztr, dim=1)         # L2-normalize -> cosine distance
    Zte = F.normalize(Zte, dim=1)
    C = int(ytr.max()) + 1
    correct = 0
    for i in range(Zte.size(0)):
        sims = Zte[i] @ Ztr.t()           # (Ntrain,) cosine similarities
        vals, idx = sims.topk(k)          # k nearest neighbors
        w = (vals / temperature).exp()    # closer neighbors vote stronger
        votes = torch.zeros(C)
        votes.index_add_(0, ytr[idx], w)  # weighted class votes
        correct += int(votes.argmax() == yte[i])
    return correct / Zte.size(0)

knn_acc = knn_eval(Ztr, ytr, Zte, yte, k=20)
print(f"kNN (k=20) accuracy:   {knn_acc:.3f}")

# To run a SEMI-SUPERVISED label-fraction curve, repeat the linear probe
# above on a random 1% / 10% / 100% subset of (Ztr, ytr), averaging over
# several seeds, and plot accuracy vs the labeled fraction.

## Visualize the data & results

_On real digit images, does a frozen representation (linear probe and kNN) need far fewer labels than training a classifier from scratch on raw pixels?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

digits = load_digits()                       # 1797 real 8x8 handwritten digits
X = digits.data / 16.0                        # pixels scaled to 0..1
y = digits.target

# hold out a fixed test set both protocols are judged on
Xrest, Xte, yrest, yte = train_test_split(X, y, test_size=0.30,
                                          random_state=0, stratify=y)
# split the rest: an ABUNDANT source pool (pretraining) + a disjoint label pool
Xsrc, Xpool, ysrc, ypool = train_test_split(Xrest, yrest, test_size=0.6,
                                            random_state=1, stratify=yrest)

# PRETRAIN a small MLP on the abundant source labels, then FREEZE its hidden layer
# (stands in for a self-supervised encoder trained on unlabeled data)
bb = MLPClassifier(hidden_layer_sizes=(64,), max_iter=600,
                   random_state=0, alpha=1e-3).fit(Xsrc, ysrc)

def features(A):                             # forward pass through the FROZEN hidden layer
    a = A
    for W, b in zip(bb.coefs_[:-1], bb.intercepts_[:-1]):
        a = np.maximum(0, a @ W + b)         # ReLU activations = the representation
    return a

Fpool, Fte = features(Xpool), features(Xte)
sc = StandardScaler().fit(Fpool)             # scale features for kNN (distance is scale-sensitive)
Fpool_s, Fte_s = sc.transform(Fpool), sc.transform(Fte)

fracs = [0.01, 0.05, 0.10, 0.25, 1.00]
probe, knn, scratch = [], [], []
for f in fracs:
    pa, ka, sa = [], [], []
    for seed in range(15):                    # average over many label draws
        rs = np.random.default_rng(seed)
        idx = []                              # stratified labeled subset of fraction f
        for c in np.unique(ypool):
            ci = np.where(ypool == c)[0]
            k = max(1, int(round(f * len(ci))))
            idx.extend(rs.choice(ci, size=min(k, len(ci)), replace=False))
        idx = np.array(idx); yk = ypool[idx]
        if len(np.unique(yk)) < 2:
            continue
        # LINEAR PROBE: logistic head on FROZEN MLP features
        pa.append(LogisticRegression(max_iter=400, solver="liblinear").fit(Fpool[idx], yk).score(Fte, yte))
        # kNN on the same frozen features
        kk = min(5, len(idx))
        ka.append(KNeighborsClassifier(n_neighbors=kk).fit(Fpool_s[idx], yk).score(Fte_s, yte))
        # FROM SCRATCH: logistic on raw pixels (no frozen representation)
        sa.append(LogisticRegression(max_iter=400, solver="liblinear").fit(Xpool[idx], yk).score(Xte, yte))
    probe.append(round(float(np.mean(pa)), 3))
    knn.append(round(float(np.mean(ka)), 3))
    scratch.append(round(float(np.mean(sa)), 3))

print("fracs  ", [int(f * 100) for f in fracs])  # [1, 5, 10, 25, 100]
print("probe  ", probe)    # [0.807, 0.927, 0.934, 0.956, 0.969]
print("knn    ", knn)      # [0.235, 0.882, 0.927, 0.954, 0.972]
print("scratch", scratch)  # [0.640, 0.840, 0.883, 0.922, 0.954]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Encoder A beats encoder B on the linear probe, but B beats A after full fine-tuning. Which encoder is "better", and what does each result tell you?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what the linear probe measures. — _It freezes the encoder and trains only a linear head, so it scores the linear separability of the frozen features. A's frozen features are more linearly separable._
- Recall what fine-tuning measures. — _It unfreezes the encoder, so it scores the best accuracy reachable when the weights can adapt. B's encoder reshapes into something better once unlocked._
- Decide by deployment plan, not by one number. — _If you will deploy frozen features, A is better. If you will fine-tune, B is better. Neither protocol is "the truth" alone._

**Answer:** It depends on how you will use the encoder. A has more linearly separable frozen features (deploy A if frozen); B adapts better when fine-tuned (deploy B if fine-tuning). Report both, because they answer different questions.

</details>

**Problem 2.** Your 1%-label point on the label-fraction curve jumps around by several accuracy points each time you rerun. Is the encoder unstable? What should you do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Diagnose the source of the noise. — _At 1% labels there are only a handful of examples per class. One unlucky random draw of which examples are labeled can swing the score a lot — this is sampling noise, not encoder instability._
- Average over many random label draws and seeds. — _Reporting the mean over (say) 30 draws plus the spread turns a noisy single number into a stable estimate of $A(\rho)$._
- Use a large, fixed test set. — _A small validation set adds its own noise; a big held-out set sharpens every point on the curve._

**Answer:** It's sampling noise from tiny labeled subsets, not an unstable encoder. Average accuracy over many random label draws (and seeds), report the spread, and evaluate on a large fixed test set.

</details>